# ML02 · 线性回归与梯度下降

用最小的线性模型，把「定义模型、量误差、靠梯度逐步修正参数」这个训练循环完整走一遍，创建后面所有神经网络的缩影。

## 学习目标
- 理解线性模型（linear model，y = wx + b）的意义，知道参数 w 与 b 各自控制什么。
- 能用均方误差（mean squared error，MSE）量化预测与真实值的差距，并理解它是要被降低的「损失（loss）」。
- 创建梯度下降（gradient descent）的直觉：为什么顺着梯度反方向走可以降低损失。
- 理解学习率（learning rate）对收敛的影响，能说明太大与太小各会发生什么。
- 能手刻一次完整的参数更新循环（前向计算、算损失、算梯度、更新参数）。
- 能用 sklearn 的 LinearRegression 对照验证手刻结果，确认方向正确。

In [ ]:
# 概念：画图前先设置 matplotlib，让中文标题不会变成方框
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 依序试常见中文字体，挑到哪个系统有就用哪个（不同操作系统字体名称不同）
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "Heiti TC", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 避免负号显示成乱码

print("matplotlib 版本:", matplotlib.__version__)
print("numpy 版本:", np.__version__)

## 线性模型是什么

线性模型（linear model）就是「一条可以调整的直线」：y = wx + b。
- 权重（weight，w）是斜率，控制 x 每增加一单位、y 跟着变多少。
- 偏置（bias，b）是截距，控制这条线整体往上或往下平移。

为什么先学它：把模型看成「一组参数（w, b）」，之后所有的学习其实都是在调这组参数，让预测（prediction）更贴近真实值。线性模型是这个观念最小、最干净的起点。

形状：y_pred = w * x + b，其中 x 是一批输入，w、b 是两个纯量。

In [ ]:
# 概念：把线性模型看成一条由参数 (w, b) 决定的直线，手动试几组参数看拟合好坏
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # 固定乱数种子，让每次运行的杂讯一致，方便对照

# 造一批带杂讯的 (x, y) 点：仿真一条真实关系 y = 2x + 5，再加上随机杂讯
x = np.linspace(0, 10, 30)                       # 0 到 10 取 30 个等距点当输入
true_w, true_b = 2.0, 5.0                         # 假设背后真正的斜率与截距
noise = rng.normal(0, 2.0, size=x.shape)          # 常态杂讯，仿真量测误差
y = true_w * x + true_b + noise

def predict(x, w, b):
    # 线性模型的前向计算：一次算出所有点的预测值（矢量化，不用循环）
    return w * x + b

# 手动试两组参数：一组刻意偏掉、一组接近真值
guess_bad = predict(x, w=1.0, b=0.0)
guess_good = predict(x, w=2.0, b=5.0)

plt.figure(figsize=(6, 4))
plt.scatter(x, y, label="数据点", color="gray")
plt.plot(x, guess_bad, label="w=1, b=0（偏掉）", color="orange")
plt.plot(x, guess_good, label="w=2, b=5（较贴）", color="green")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("不同参数画出不同的线")
plt.show()

print("前 3 点的真实 y:", np.round(y[:3], 2))
print("w=2,b=5 的预测:", np.round(guess_good[:3], 2))

## 用 MSE 量误差

肉眼判断「哪条线比较贴」不够客观，需要一个数字代表「整体有多错」。这个数字就是损失（loss）。

均方误差（mean squared error，MSE）是最常用的回归损失：把每个点的残差（residual，预测值减真实值）平方后取平均。平方让正负误差不会互相抵消，也让大误差被放大。MSE 越小代表线越贴，这就是我们要最小化的目标。

公式：MSE = 平均（(y_pred − y_true)²）。

In [ ]:
# 概念：写一个 mse 函数把整体误差浓缩成一个数字，比较不同参数的好坏
import numpy as np

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)   # 与上一节相同的自造数据

def mse(y_true, y_pred):
    residual = y_pred - y_true     # 残差：每个点预测偏离真实多少
    return np.mean(residual ** 2)  # 平方后取平均，得到单一损失数字

def predict(x, w, b):
    return w * x + b

# 比较三组参数的 MSE：越接近真值 (w=2, b=5) 应该越小
for w, b in [(1.0, 0.0), (2.0, 0.0), (2.0, 5.0)]:
    loss = mse(y, predict(x, w, b))
    print(f"w={w}, b={b}  ->  MSE = {loss:.3f}")

## 梯度下降的直觉

我们想找一组 (w, b) 让 MSE 最小，但不可能无限地乱试。梯度下降（gradient descent）给出有方向的找法。

把损失想成一座山，山的高度就是 MSE，脚下的位置就是当前参数。梯度（gradient）指向「上坡最陡」的方向，所以往梯度的反方向走一小步，损失就会下降，这就是「下山」的比喻。整个损失随参数变化的形状称为损失曲面（loss surface）。

关键直觉：对 w 的梯度若为正，代表往 w 变大会让损失上升，所以该把 w 调小；为负则反之。

In [ ]:
# 概念：固定 b、只扫描 w，画出损失的碗形曲线并标出当前点与下山方向
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)

def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

fixed_b = 5.0
w_grid = np.linspace(-1, 5, 100)                          # 扫描一段 w 值
losses = [mse(y, w * x + fixed_b) for w in w_grid]        # 每个 w 对应一个损失，组成碗形曲线

current_w = 0.5                                            # 假设目前站在这个 w
# 对 w 的梯度（MSE 对 w 微分后的解析式）：方向告诉我们该往哪走
current_grad = np.mean(2 * (current_w * x + fixed_b - y) * x)
current_loss = mse(y, current_w * x + fixed_b)

plt.figure(figsize=(6, 4))
plt.plot(w_grid, losses, label="损失随 w 变化")
plt.scatter([current_w], [current_loss], color="red", zorder=5, label="目前位置")
# 梯度为正代表上坡在右边，下山要往左（w 变小）；画一个指向下坡的箭头示意
plt.annotate("", xy=(current_w - 0.8, current_loss), xytext=(current_w, current_loss),
             arrowprops=dict(arrowstyle="->", color="red"))
plt.xlabel("w"); plt.ylabel("MSE"); plt.legend(); plt.title("损失曲面（固定 b 只看 w）")
plt.show()

print(f"目前 w={current_w} 的梯度 = {current_grad:.2f}（正数代表该把 w 调小才会下山）")

## 学习率与收敛

梯度只给「方向」，但每一步要走多大由学习率（learning rate）决定，这个步幅也称为步长（step size）。

口诀是「方向由梯度给，幅度由学习率给」。学习率太小，每步挪一点点，要很多步才到谷底（收敛 convergence 很慢）；太大则可能一步跨过谷底、在两侧来回弹，甚至越跑越远而发散（divergence）。挑一个适中的学习率，是训练能不能成功的关键。

In [ ]:
# 概念：用同一笔数据跑三种学习率，比较损失随迭代下降的曲线（收敛 vs 发散）
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)
n = len(x)

def run(lr, steps=40):
    w, b = 0.0, 0.0                              # 都从同一个起点出发，公平比较
    history = []
    for _ in range(steps):
        y_pred = w * x + b
        history.append(np.mean((y_pred - y) ** 2))
        # 梯度：MSE 分别对 w、b 微分得到的解析式
        grad_w = (2 / n) * np.sum((y_pred - y) * x)
        grad_b = (2 / n) * np.sum(y_pred - y)
        w -= lr * grad_w                          # 往梯度反方向走，步幅由 lr 控制
        b -= lr * grad_b
    return history

plt.figure(figsize=(6, 4))
# 注意：过大的学习率会让损失不减反增，曲线往上甚至爆掉
for lr, name in [(0.0005, "过小 lr=0.0005"), (0.01, "适中 lr=0.01"), (0.025, "过大 lr=0.025")]:
    plt.plot(run(lr), label=name)
plt.xlabel("迭代次数"); plt.ylabel("MSE"); plt.legend(); plt.title("不同学习率的下降曲线")
plt.yscale("log")   # 技巧：用对数刻度，让收敛与发散的差距在同一张图看得清楚
plt.show()

## 手刻训练循环

现在把前面的零件串成一个完整的训练循环。每一轮（迭代 iteration，扫过全部数据一次也称为一个 epoch）做四件事：
1. 前向计算（forward pass）：用目前的 w、b 算出预测。
2. 算损失：用 MSE 看现在有多错。
3. 算梯度：得到每个参数该往哪调。
4. 参数更新（parameter update）：往梯度反方向各走一步。

重复数十次直到损失几乎不再下降，就近似收敛。这个循环就是后面所有神经网络训练的缩影，差别只在模型更大、参数更多。

In [ ]:
# 概念：把前向、损失、梯度、更新串成一个 for 循环，从随机起点迭代出最终 w、b
import numpy as np

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)   # 真值 w=2, b=5，看循环能不能逼近
n = len(x)

w, b = rng.normal(0, 1), rng.normal(0, 1)   # 随机初始参数，仿真「一无所知的起点」
lr = 0.01
epochs = 60

for epoch in range(epochs):
    y_pred = w * x + b                         # 1. 前向计算
    loss = np.mean((y_pred - y) ** 2)          # 2. 算损失
    grad_w = (2 / n) * np.sum((y_pred - y) * x)  # 3. 算梯度
    grad_b = (2 / n) * np.sum(y_pred - y)
    w -= lr * grad_w                           # 4. 更新参数（同时更新 w 与 b）
    b -= lr * grad_b
    if epoch % 10 == 0 or epoch == epochs - 1:   # 每隔几轮印一次，观察损失逐步下降
        print(f"epoch {epoch:2d}  loss={loss:7.3f}  w={w:.3f}  b={b:.3f}")

print("---")
print(f"最终参数: w={w:.3f}, b={b:.3f}（真值为 w=2.0, b=5.0）")

## 用 sklearn 对照验证

手刻循环得到一组 w、b，但怎么知道它对不对？用成熟工具求出「标准答案」来对照。

scikit-learn 的 LinearRegression 会用解析解（最小平方法）直接算出最佳参数：调用 fit 拟合数据后，用 coef_ 取得系数（斜率）、用 intercept_ 取得截距。它背后做的事与手刻循环一致，只是把流程封装起来。若手刻结果与它接近，就代表方向正确。

In [ ]:
# 概念：用 sklearn LinearRegression 求标准答案，与手刻循环的参数并排比较
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)
n = len(x)

# 先重跑一次手刻循环，拿到手刻的 w、b
w, b = 0.0, 0.0
for _ in range(2000):
    y_pred = w * x + b
    w -= 0.01 * (2 / n) * np.sum((y_pred - y) * x)
    b -= 0.01 * (2 / n) * np.sum(y_pred - y)

# 注意：sklearn 的 fit 要求特征是二维 (样本数, 特征数)，所以把一维 x 变成栏矢量
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)

print(f"手刻循环   : w={w:.4f}, b={b:.4f}")
print(f"sklearn   : w={model.coef_[0]:.4f}, b={model.intercept_:.4f}")
print(f"两者差距   : dw={abs(w - model.coef_[0]):.4f}, db={abs(b - model.intercept_):.4f}")

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）用一个特征预测房价（集成：线性模型 + MSE）
#   情境：自造一批公寓数据，用「楼地板面积」单一特征预测房价（仿真线性加杂讯）。
#   要求：
#     1. 用 numpy 自造 (面积, 房价) 数据，并手选一组 w、b 用 predict 画出预测线。
#     2. 写 mse 函数，调整 w、b 试着把 MSE 压到更低，记下最好的一组并印出。
#   验收：应该看到 MSE 数值随着参数调整而下降，且预测线明显更贴近数据点。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）手刻梯度下降拟合容积率（集成：MSE + 梯度下降 + 学习率 + 训练循环）
#   情境：自造一批基地数据，用「建蔽率」预测「容积率」（仿真线性关系）。
#   要求：
#     1. 从随机初始 w、b 开始，写一个梯度下降循环逐步更新参数。
#     2. 记录每次迭代的 MSE，画出损失下降曲线。
#     3. 换一个明显过大的学习率重跑，观察曲线变化。
#   验收：应该看到适中学习率下损失平滑下降并收敛，而过大学习率下损失震荡或发散。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）手刻与 sklearn 双轨验证并调学习率（集成：训练循环 + 学习率 + sklearn 对照 + 结果分析）
#   情境：自造一批街廓数据，用「临路宽度」预测「平均楼高」（仿真线性加杂讯）。
#   要求：
#     1. 手刻梯度下降循环求 w、b，并尝试至少三种学习率挑出最佳收敛者。
#     2. 用 sklearn LinearRegression 对同一数据求标准答案（记得把 x reshape 成二维）。
#     3. 比较手刻与 sklearn 的参数差距，并用注解简述：若差距偏大，可能是学习率或迭代次数哪里需要调整。
#   验收：应该看到手刻参数与 sklearn 结果接近，且能说出让两者更一致的调整方向。
# TODO: 在这里完成


## 小结

- 线性模型是一条由参数 (w, b) 决定的直线：w 是斜率、b 是截距；学习的本质就是调这组参数。
- MSE 把每点残差平方再平均，用一个数字代表整体误差，是要被最小化的损失。
- 梯度下降把损失当成一座山，往梯度反方向走一小步就会下山，逐步逼近最低点。
- 学习率决定每步的幅度：太小收敛慢、太大会震荡甚至发散；方向由梯度给、幅度由学习率给。
- 手刻训练循环就是「前向计算、算损失、算梯度、更新参数」重复数十次，这正是神经网络训练的缩影。
- 用 sklearn LinearRegression 求出的系数与截距可当标准答案，验证手刻结果方向正确。